# LLM-RecFusion — Stage 5: 极限压榨显存——AWQ 模型量化实战

**Objective**: 让重排大模型（Qwen-1.8B）在显存极度受限（≈6GB VRAM）的边缘设备或消费级显卡上流畅运行。
通过 AWQ (Activation-aware Weight Quantization) INT4 量化，将模型显存占用从 ~3.6GB 打爆到 ~0.9GB。

---

### 量化路线图

```
原始 FP16 权重 (16-bit)    →    AWQ INT4 量化权重 (4-bit)
      ↓                               ↓
  ~3.6 GB VRAM                  ~0.9 GB VRAM
      ↓                               ↓
  高精度、高显存              近乎无损精度、极低显存
  6GB 显卡跑不动             6GB 显卡轻松部署
```

---

### Notebook 流程

```
┌─────────────────────────────────────────────────────────────────────┐
│ 1. 环境与依赖准备 → AutoAWQ + Transformers                         │
├─────────────────────────────────────────────────────────────────────┤
│ 2. 核心量化逻辑 → quant_config + model.quantize() + save_quantized │
│    多行中文注解：q_group_size=128 为何是黄金分割点？               │
├─────────────────────────────────────────────────────────────────────┤
│ 3. 极客对决：显存算账 → FP16 vs INT4 柱状图（暗色主题）            │
├─────────────────────────────────────────────────────────────────────┤
│ 4. 极客硬核笔记 → 面试杀手锏：AWQ 为什么碾压 RTN？                  │
└─────────────────────────────────────────────────────────────────────┘
```

---

## 1. 环境与依赖准备

导入核心库。如果当前环境缺少 GPU 或 `autoawq` 依赖，Notebook 仍能优雅降级完成文字和图表渲染。

In [ ]:
# ============================================================
# 1. 环境与依赖准备
# ============================================================

import sys
import os
import math
import warnings
from typing import Optional, Dict, Any

warnings.filterwarnings('ignore')

# ---------- AWQ 量化核心库 ----------
# AutoAWQ: 业界标准 INT4 量化框架，支持 Activation-aware 权重保护
try:
    from awq import AutoAWQForCausalLM
    from awq.utils.utils import clear_memory
    AWQ_AVAILABLE = True
    print("[✓] AutoAWQ 导入成功")
except ImportError:
    AWQ_AVAILABLE = False
    print("[!] AutoAWQ 未安装。量化核心功能不可用，但本 Notebook 仍会展示代码模板和显存计算。")
    print("    安装命令: pip install autoawq")

# ---------- Transformers 库 ----------
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    print("[✓] Transformers 导入成功")
except ImportError:
    print("[!] Transformers 未安装: pip install transformers")

# ---------- 可视化 ----------
try:
    import matplotlib.pyplot as plt
    import matplotlib
    print("[✓] Matplotlib 导入成功")
except ImportError:
    print("[!] Matplotlib 未安装: pip install matplotlib")

# ---------- 数据计算 ----------
import numpy as np

print("\n[✓] 环境准备完成")
print(f"    Python 版本: {sys.version}")

In [ ]:
# ============================================================
# 占位符路径配置（按需替换为实际路径）
# ============================================================

# HuggingFace 上的模型名称（首次运行会自动下载，约 3.5GB）
# 也可以替换为本地下载好的模型路径，例如 ./models/Qwen1.5-1.8B
model_path: str = "Qwen/Qwen1.5-1.8B"

# 量化后模型保存路径
quant_path: str = "models/Qwen1.5-1.8B-AWQ-Int4"

print(f"[•] 原始模型路径: {model_path}")
print(f"[•] 量化保存路径: {quant_path}")

## 2. 核心量化逻辑实现

AWQ 量化的精髓在于：**不是简单地砍掉精度，而是保护 1% 的显著权重不被误伤**。
下面的 `quant_config` 中的每一个参数都有其物理意义，尤其是 `q_group_size=128`。

In [ ]:
# ============================================================
# 2. 核心量化逻辑：配置 -> 加载 -> 量化 -> 保存
# ============================================================

# ------------------------------------------------------------------
# 量化配置字典 —— 每一个参数都有讲究
# ------------------------------------------------------------------
quant_config: Dict[str, Any] = {
    "zero_point": True,      # 零点量化：让量化网格可以偏移，更好地适配非对称分布的张量
                              # 如果不设 zero_point，相当于强制分布对称，精度会下降 1~2%

    "q_group_size": 128,     # ================================
                              # 【核心参数 - 面试必问】
                              # q_group_size 即【分组量化粒度】：将每 128 个权重分为一组，
                              # 每组独立计算自己的缩放因子 (scale) 和零点 (zero_point)。
                              #
                              # * 为什么是 128？
                              #   设得太小（如 32）：分组过细，每个组的 scale 存储开销暴增，
                              #   总模型体积反而变大，抵消 INT4 的压缩收益。
                              #   设得太大（如 512 或 1024）：分组太粗，组内权重分布差异大，
                              #   量化误差累积，精度断崖式下降。
                              #
                              # * 128 是黄金分割点：
                              #   经过大量实验验证，128 在压缩率（4-bit 量化）和精度维持
                              #   （PPL / 下游任务 <0.5% 损失）之间达到了最优帕累托前沿。
                              #   这是学术界和工业界（包括 Llama.cpp, AutoAWQ, GPTQ）
                              #   共同认可的标准值。
                              # ================================

    "w_bit": 4,              # 权重量化位宽：4-bit
                              # FP16(16-bit) -> INT4(4-bit) = 4x 显存压缩
                              # 理论：1.8B x 2 bytes -> 1.8B x 0.5 bytes = 0.9GB

    "version": "GEMM",      # 量化核函数版本：GEMM (General Matrix Multiply)
                              # GEMM 适合批量推理，吞吐更高；GEMV 适合单流低延迟场景。
                              # 对重排场景（批量打分），GEMM 是更优选择。
}

print("[✓] 量化配置已定义")
for k, v in quant_config.items():
    print(f"    {k:20s} = {v}")

In [ ]:
# ============================================================
# 模型加载、量化与保存（带 Try-Except 保护）
# ============================================================

# 如果当前 GPU 显存不足 6GB，建议不要直接运行此 cell，
# 或者将 model_path 指向一个更小的模型（如 0.5B）。

if AWQ_AVAILABLE:
    try:
        print("[1/3] 正在加载模型和 Tokenizer ...")
        print(f"      模型: {model_path}")
        print("      首次加载会从 HuggingFace 下载 ~3.5GB 权重，请耐心等待...")

        # 加载量化感知模型
        model = AutoAWQForCausalLM.from_pretrained(
            model_path,
            device_map="auto",       # 自动分配到可用 GPU
            trust_remote_code=True,   # 安全起见，Qwen 等模型需要此选项
        )

        # 加载对应的 Tokenizer
        tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            trust_remote_code=True,
        )
        print("[✓] 模型加载成功")

        # ---------- 准备校准数据 ----------
        # 校准数据是 AWQ 的核心：通过少量样本观察激活分布，找出显著权重
        # 实际生产中通常从训练数据中采样 128~512 条
        calib_text: list = [
            "深度学习推荐系统中，用户行为序列建模是核心挑战之一。",
            "LLM 在推荐领域的应用正从 Pointwise 打分向 Listwise 生成式排序演进。",
            "模型量化是将大模型部署到边缘设备的关键技术。",
            "AWQ 通过保护 1% 的显著权重，实现了 INT4 下的近乎无损精度。",
        ]

        print("\n[2/3] 正在执行 AWQ INT4 量化 ...")
        print("      使用校准数据观察激活分布，识别 Salient Weights...")

        # ---------- 核心量化 API ----------
        model.quantize(
            tokenizer,
            quant_config=quant_config,
            calib_data=calib_text,   # 校准数据用于分析激活分布
        )
        print("[✓] INT4 量化完成！")

        # ---------- 保存量化模型 ----------
        print(f"\n[3/3] 正在保存量化模型到: {quant_path}")
        os.makedirs(os.path.dirname(quant_path), exist_ok=True)

        model.save_quantized(
            quant_path,
            safetensors=True,         # 使用 safetensors 格式（更安全、更快）
            shard_size="1GB",        # 如果模型较大，分片存储
        )
        tokenizer.save_pretrained(quant_path)
        print(f"[✓] 量化模型已保存至: {quant_path}")

        # ---------- 显存清理 ----------
        clear_memory()
        print("\n[✓] 量化全流程完成！现在可以用 ~0.9GB 显存运行 Qwen-1.8B 了。")

    except Exception as e:
        print(f"[!] 量化过程出现异常: {e}")
        print("    可能原因：")
        print("      - 当前环境没有 GPU（CUDA 不可用）")
        print("      - GPU 显存不足（需要至少 ~6GB 加载 FP16 模型进行量化）")
        print("      - 网络问题导致模型下载失败")
        print("    建议：在 Colab A100 / V100 或本地 RTX 3060+ 上运行此 cell")
        print("    本 Notebook 剩余的文字和图表部分不受影响。")
else:
    print("[!] AutoAWQ 未安装，跳过量化执行。")
    print("    量化核心逻辑代码在本文档中已完整展示，可直接用于生产环境。")

---

## 3. 极客对决：显存算账（VRAM Benchmark）

理论上的显存占用计算公式：

$$
VRAM = (param_count * bit_width) / 8 (Bytes)
$$

| 精度 | 每参数位宽 | 1.8B 模型显存 | 对比 |
|------|-----------|---------------|------|
| FP16 | 16 bit    | 1.8e9 x 2 = 3.6 GB | 基准 |
| INT4 | 4 bit     | 1.8e9 x 0.5 = 0.9 GB | **4x 压缩!** |

> **注意**：上述仅为**权重**显存。实际推理还需加上 KV Cache（~0.5~1GB 取决于序列长度）和激活值（~0.3~0.5GB），
> 因此 FP16 下实际需要 ~5~6GB，在 6GB 显卡上捉襟见肘；INT4 下仅需 ~1.7~2GB，绰绰有余。

In [ ]:
# ============================================================
# 3. 辅助函数：显存计算 + 暗色主题柱状图
# ============================================================

def calculate_vram_usage(param_count_billions: float) -> Dict[str, float]:
    """
    计算给定参数量模型在不同精度下的理论显存占用。

    Parameters
    ----------
    param_count_billions : float
        模型参数量，单位为 Billion（十亿）。例如 Qwen-1.8B 传 1.8。

    Returns
    -------
    Dict[str, float]
        包含 FP16 和 INT4 的显存占用（单位：GB）
    """
    params = param_count_billions * 1e9

    # FP16: 每个参数 16 bit = 2 bytes
    fp16_bytes = params * 2
    fp16_gb = fp16_bytes / (1024 ** 3)

    # INT4: 每个参数 4 bit = 0.5 bytes
    int4_bytes = params * 0.5
    int4_gb = int4_bytes / (1024 ** 3)

    return {"FP16": round(fp16_gb, 2), "INT4": round(int4_gb, 2)}


# ---------- 算账：1.8B 模型 ----------
MODEL_PARAMS: float = 1.8  # Qwen1.5-1.8B
vram_dict = calculate_vram_usage(MODEL_PARAMS)

print("=" * 55)
print("   🔥  显 存 对 决 ：  FP16  vs  INT4")
print("=" * 55)
print(f"   模型参数量: {MODEL_PARAMS}B")
print()
print(f"   🟦  FP16 (16-bit)   ->  {vram_dict['FP16']:>6.2f} GB    <- 原始精度")
print(f"   🟩  INT4  (4-bit)   ->  {vram_dict['INT4']:>6.2f} GB    <- AWQ 量化")
print()
print(f"   📉 压缩比: {vram_dict['FP16'] / vram_dict['INT4']:.1f}x")
print(f"   💾 节省显存: {vram_dict['FP16'] - vram_dict['INT4']:.2f} GB")
print("=" * 55)

# ---------- 补充：加上 KV Cache 估算后的实际占用 ----------
# 重排场景典型序列长度 512，4 个候选，估算 KV Cache ~0.8GB
KV_CACHE_ESTIMATE_GB: float = 0.8
ACTIVATIONS_ESTIMATE_GB: float = 0.4

fp16_real = vram_dict['FP16'] + KV_CACHE_ESTIMATE_GB + ACTIVATIONS_ESTIMATE_GB
int4_real = vram_dict['INT4'] + KV_CACHE_ESTIMATE_GB + ACTIVATIONS_ESTIMATE_GB

print()
print("   📊 实际推理显存估算（权重 + KV Cache + 激活值）:")
print(f"   🟦  FP16 实际占用: {fp16_real:.2f} GB  ->  6GB 显卡 {'❌ 超出!' if fp16_real > 6 else '✅ 勉强'}")
print(f"   🟩  INT4 实际占用: {int4_real:.2f} GB  ->  6GB 显卡 {'✅ 轻松运行' if int4_real < 6 else '⚠️ 接近极限'}")

In [ ]:
# ============================================================
# 极客暗色主题 - 显存对比柱状图
# ============================================================

# ---------- 全局暗色主题配置 ----------
plt.style.use('dark_background')

fig, ax = plt.subplots(figsize=(8, 5), facecolor='#1a1a2e')
ax.set_facecolor('#16213e')

# ---------- 数据 ----------
categories = ['FP16 (16-bit)', 'INT4 (4-bit)']
values = [vram_dict['FP16'], vram_dict['INT4']]
colors = ['#e94560', '#0f3460']  # 霓虹红 vs 深空蓝

# ---------- 柱状图 ----------
bars = ax.bar(
    categories,
    values,
    color=colors,
    width=0.5,
    edgecolor='white',
    linewidth=1.2,
    alpha=0.9,
)

# ---------- 柱顶标注 ----------
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.08,
        f'{val:.2f} GB',
        ha='center',
        va='bottom',
        fontsize=14,
        fontweight='bold',
        color='white',
    )

# ---------- 添加压缩比箭头 ----------
ax.annotate(
    f'4.0x 压缩!',
    xy=(0, values[0]),
    xytext=(1, values[1] + 0.6),
    fontsize=13,
    fontweight='bold',
    color='#ffd700',  # 金色
    arrowprops=dict(
        arrowstyle='->',
        color='#ffd700',
        lw=2.5,
        connectionstyle='arc3,rad=-0.3',
    ),
)

# ---------- 标题与标签 ----------
ax.set_title(
    '⚡ 显存占用对比 · Qwen-1.8B 权重 (仅权重)'
    '\nAWQ INT4 量化后显存断崖式下降',
    fontsize=15,
    fontweight='bold',
    color='white',
    pad=20,
)
ax.set_ylabel('显存占用 (GB)', fontsize=12, color='white')
ax.set_ylim(0, max(values) * 1.5)

# ---------- 网格线 ----------
ax.grid(axis='y', alpha=0.3, linestyle='--', color='gray')
ax.set_axisbelow(True)

# ---------- 边框隐藏 ----------
for spine in ax.spines.values():
    spine.set_visible(False)

# ---------- Tick 颜色 ----------
ax.tick_params(colors='white', labelsize=11)

plt.tight_layout()
plt.savefig('images/awq_vram_benchmark.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print("[✓] 图表已保存至: images/awq_vram_benchmark.png")

---

## 4. 极客硬核笔记：面试杀手锏素材

### 为什么要用 AWQ 而不是基础的 RTN (Round-To-Nearest) 量化？

---

#### 【核心洞察】权重并非生而平等

在深度学习模型中，**并不是所有权重都同等重要**。

对激活值分布影响最大的，仅仅是模型中 **约 1% 的显著权重（Salient Weights）**。
这些权重的微小扰动会通过残差连接逐层放大，最终显著改变整个模型的输出分布。

```
████████████████████████████████  100% 权重
███                                   ~ 1% 显著权重
└────── 这 1% 决定了激活层 90% 以上的分布质量
```

---

#### 【RTN 为什么不行？】

RTN（Round-To-Nearest，最近邻舍入）是最朴素的量化方法：直接对每个权重做四舍五入到最近的量化网格点。

**RTN 的三个致命问题：**

1. **一视同仁，不分轻重** -- 1% 的显著权重和 99% 的普通权重享受同样的舍入误差。
2. **误差累积效应** -- 每个权重的微小舍入误差（+-0.5bit）经过多层线性变换和激活函数后，
   在输出层可能放大为 5~10% 的精度损失。
3. **校准数据零利用** -- RTN 完全不看数据，不做激活分布分析，属于【盲人摸象】式量化。

```
RTN 量化的命运:          AWQ 量化的命运:

  权重 -> 直接舍入          权重 -> 分析激活分布 -> 识别显著权重
    |                              |
  精度损失 3~8%              保护显著权重 -> 缩放调整
    |                              |
  INT4 省显存但不可用        INT4 省显存 + 精度近乎保留
```

---

#### 【AWQ 为什么是突破？】

AWQ（Activation-aware Weight Quantization）的核心突破在于：

| 维度 | RTN | AWQ |
|------|-----|-----|
| 是否使用校准数据 | ❌ 不用 | ✅ 用少量文本观察激活分布 |
| 是否区分显著权重 | ❌ 不区分 | ✅ 精准定位 ~1% 的 Salient Weights |
| 对显著权重的处理 | 同等待遇，粗暴舍入 | 缩放保护 + 分组调整，避免量化误差 |
| INT4 精度损失 | 3~8% 任务精度下降 | < 0.5% 任务精度下降 |
| 推理速度 | 较快（但精度不可用） | 快（GEMM 优化，接近 RTN 速度） |
| 工业落地可行性 | ❌ 低（精度不可接受） | ✅ **高 - 业界标准方案** |

---

#### 【面试官最爱问：一句话总结】

> **AWQ 通过观察校准数据的激活分布，精准识别并保护那 1% 决定模型命运的显著权重不被粗暴量化，
> 从而在享受 INT4 极低显存占用（4x 压缩）的同时，保留了近乎 FP16 的大模型推理精度。
> 这是突破硬件算力天花板的最优雅解法。**

---

#### 【参考文献】

- Lin, J., Tang, J., Tang, H., et al. *AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration*. arXiv:2306.00978, 2023.
- 业界实现: [AutoAWQ](https://github.com/casperhansen/AutoAWQ) -- MIT License
- 核心思想: 不是所有权重生而平等 -- 约 1% 的权重决定了 90% 以上的激活质量。

---

## 总结

| 模块 | 关键产出 |
|------|---------|
| **1. 环境** | AutoAWQ + Transformers 依赖准备，Try-Except 保护 |
| **2. 量化逻辑** | quant_config + model.quantize() + model.save_quantized() 完整模板 |
| **3. 显存算账** | FP16 3.6GB -> INT4 0.9GB，4x 压缩，暗色柱状图 |
| **4. 硬核笔记** | AWQ > RTN 的底层原理：1% 显著权重保护机制 |

**下一步**：在 `notebooks/17_rerank_inference_engine.ipynb` 中，我们将加载这个量化后的 INT4 模型，
构建 Listwise 生成式重排推理引擎，实现边缘设备上的毫秒级重排。